In [19]:
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

Keep only labels for stress and neutral

In [20]:
features = pd.read_csv("../Data/wesad_eda/feature_dataframe.csv")
neutral = features[features['label'] == 1]
stress = features[features['label'] == 2]
feature_binary = pd.concat([neutral, stress])
feature_binary.columns

Index(['label', 'subject_id', 'eda_mean', 'eda_max', 'eda_std', 'eda_skew',
       'eda_norm1', 'phasic_mean', 'phasic_max', 'phasic_std', 'phasic_skew',
       'phasic_norm1', 'phasic_peak_count', 'phasic_std_rise_time',
       'tonic_mean', 'tonic_max', 'tonic_std', 'tonic_skew', 'tonic_norm1'],
      dtype='object')

Apply feature wise per subject normalization

In [21]:
feature_cols = ['phasic_mean', 'phasic_peak_count', 'phasic_std_rise_time', 'tonic_mean', 'tonic_max', 'phasic_skew']

def minmax_per_subject(group):
    return (group - group.min()) / (group.max() - group.min())

feature_binary[feature_cols] = (
    feature_binary
    .groupby('subject_id')[feature_cols]
    .transform(minmax_per_subject)
)

In [22]:
feature_binary = feature_binary.dropna()
#feature_binary = feature_binary.drop(columns = ['phasic_std_rise_time'])
y = feature_binary[['label']].values.ravel()
X = feature_binary.iloc[:, 2:]


Reduced dimension is from keeping only rows with select labels.

In [23]:
print(X.shape)
print(y.shape)

(843, 17)
(843,)


Run Adaboost

In [24]:
groups = feature_binary['subject_id']

logo = LeaveOneGroupOut()

param_grid = {
    'n_estimators': [200, 250, 350, 400],
    'learning_rate': [0.1, 0.05, 0.001, 0.005],
    'estimator__max_depth': [1, 2, 3], 
}

grid = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier()),
    param_grid=param_grid,
    cv=logo,
    scoring='accuracy')

grid.fit(X, y, groups=groups)  # groups= is required here, easy to forget

print(f"Best params: {grid.best_params_}")
print(f"Best LOGO accuracy: {grid.best_score_:.4f}")

Best params: {'estimator__max_depth': 2, 'learning_rate': 0.001, 'n_estimators': 400}
Best LOGO accuracy: 0.8928


In [25]:
model = AdaBoostClassifier(n_estimators=50, learning_rate=1)
logo = LeaveOneGroupOut()
# Run the Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# 4. Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.88135593 0.59649123 1.         0.64912281 1.         0.92982456
 0.56896552 0.83636364 0.7037037  1.         1.         1.
 0.92857143 0.96491228 1.        ]
Average Accuracy across all unseen subjects: 0.8706


In [26]:
# Initialize Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
# Run Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.88135593 0.66666667 1.         0.71929825 1.         0.92982456
 0.63793103 0.87272727 0.68518519 1.         0.98214286 1.
 0.98214286 0.98245614 1.        ]
Average Accuracy across all unseen subjects: 0.8893


In [27]:
# Initialize Support Vector Classifier (SVC)
model = SVC(kernel='rbf', C=1.0, random_state=42)

# Set up Leave-One-Group-Out cross-validation
logo = LeaveOneGroupOut()

# Run Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.94915254 0.61403509 1.         0.64912281 1.         1.
 0.46551724 0.78181818 0.66666667 1.         1.         1.
 0.57142857 0.96491228 0.94642857]
Average Accuracy across all unseen subjects: 0.8406
